In [3]:
import pandas as pd
import re

# =========================
# FICHIERS
# =========================
csv_file = "La-ballade-de-Pern-intégrale-sentences.csv"
ods_file = "Noms personnages de Pern.ods"
output_file = "phrases_avec_un_mot_detecte_par_ligne.csv"

# =========================
# FONCTIONS
# =========================
def normalize_text(s):
    if pd.isna(s):
        return ""
    s = str(s).strip()
    s = s.replace("’", "'").replace("`", "'")
    s = re.sub(r"\s+", " ", s)
    return s

def make_forms(name):
    """
    Génère plusieurs variantes d'un nom à exclure.
    """
    name = normalize_text(name)
    if not name:
        return set()

    forms = {name, name.lower()}

    simple_space = name.replace("'", "").replace("-", " ")
    simple_space = re.sub(r"\s+", " ", simple_space).strip()
    if simple_space:
        forms.add(simple_space)
        forms.add(simple_space.lower())

    simple_nospace = name.replace("'", "").replace("-", "")
    if simple_nospace:
        forms.add(simple_nospace)
        forms.add(simple_nospace.lower())

    return forms

def first_word_span(text):
    """
    Renvoie la position du premier vrai mot de la phrase.
    """
    if pd.isna(text):
        return None

    text = str(text)
    pattern = re.compile(r"[A-Za-zÀ-ÖØ-öø-ÿ][A-Za-zÀ-ÖØ-öø-ÿ'’-]*")
    m = pattern.search(text)
    if m:
        return m.span()
    return None

def extract_internal_capitalized_words(text, excluded_forms):
    """
    Extrait les mots commençant par une majuscule mais non placés
    au début de la phrase, en excluant les listes fournies.
    Renvoie une liste de mots.
    """
    if pd.isna(text):
        return []

    text = str(text).strip()
    if not text:
        return []

    first_span = first_word_span(text)

    pattern = re.compile(
        r"\b([A-ZÀ-ÖØ-Þ][A-Za-zÀ-ÖØ-öø-ÿ'’-]*[a-zà-öø-ÿ][A-Za-zÀ-ÖØ-öø-ÿ'’-]*)\b"
    )

    matches = []

    for m in pattern.finditer(text):
        word = m.group(1)
        span = m.span(1)

        # ignorer le premier mot de la phrase
        if first_span and span == first_span:
            continue

        word_norm = normalize_text(word)

        variants = {
            word_norm,
            word_norm.lower(),
            word_norm.replace("'", "").replace("-", " "),
            word_norm.replace("'", "").replace("-", " ").lower(),
            word_norm.replace("'", "").replace("-", ""),
            word_norm.replace("'", "").replace("-", "").lower(),
        }

        if variants & excluded_forms:
            continue

        matches.append(word)

    return matches

# =========================
# 1. CHARGER LES EXCLUSIONS
# =========================
excluded_forms = set()

sheet1 = pd.read_excel(
    ods_file,
    sheet_name="Nom et genre persos",
    engine="odf"
)

for col in ["NOMS HUMAINS", "NOMS DRAGONS"]:
    if col in sheet1.columns:
        for val in sheet1[col].dropna():
            excluded_forms.update(make_forms(val))

sheet2 = pd.read_excel(
    ods_file,
    sheet_name="Noms propres pas perso",
    engine="odf"
)

for col in ["FORTS ", "PROFESSION", "RANG", "AUTRE"]:
    if col in sheet2.columns:
        for val in sheet2[col].dropna():
            excluded_forms.update(make_forms(val))

# =========================
# 2. LIRE LE CSV
# =========================
df = pd.read_csv(csv_file)

if "text" not in df.columns:
    raise ValueError("La colonne 'text' est introuvable dans le CSV.")

# =========================
# 3. EXTRAIRE LES MOTS
# =========================
df["detected_words"] = df["text"].apply(
    lambda x: extract_internal_capitalized_words(x, excluded_forms)
)

# garder seulement les phrases avec au moins un mot détecté
result = df[df["detected_words"].apply(len) > 0].copy()

# une ligne par mot détecté
result = result.explode("detected_words")

# renommer la colonne pour qu'elle soit explicite
result = result.rename(columns={"detected_words": "mot_detecte"})

# optionnel : réinitialiser l'index
result = result.reset_index(drop=True)

# =========================
# 4. EXPORT
# =========================
result.to_csv(output_file, index=False, encoding="utf-8-sig")

print(f"Fini : {len(result)} lignes exportées.")
print(f"Résultat enregistré dans : {output_file}")

Fini : 43284 lignes exportées.
Résultat enregistré dans : phrases_avec_un_mot_detecte_par_ligne.csv
